# Define a Knowledge Graph from Scratch

End-to-end walkthrough of the genai-graph new design:

1. Define Pydantic domain models
2. Declare `GraphNode` / `GraphRelation` / `GraphSchema`
3. Inspect the compiled schema
4. Visualise the **schema** as an interactive HTML diagram
5. Ingest data into an in-memory Ladybug graph
6. Query the KG with Cypher
7. Visualise the **KG data** as an interactive HTML diagram

No external database or config required — runs fully in memory.


In [1]:
# -- Shared display helper import ---------------------------------------------
from genai_graph.utils.notebooks import show_html_in_notebook

## 1. Define Your Domain Models

Plain Pydantic models — no graph imports needed here.

In [2]:
from pydantic import BaseModel, Field


class Address(BaseModel):
    city: str
    country: str = "Unknown"


class Company(BaseModel):
    name: str
    sector: str | None = None
    hq: Address | None = None


class Person(BaseModel):
    name: str
    role: str | None = None


class Risk(BaseModel):
    description: str
    impact: str = "medium"


class Project(BaseModel):
    """Root model — the entry point for graph traversal."""

    title: str
    status: str = "active"
    client: Company
    team: list[Person] = Field(default_factory=list)
    risks: list[Risk] = Field(default_factory=list)


print("Models defined:", [m.__name__ for m in [Project, Company, Person, Risk, Address]])

Models defined: ['Project', 'Company', 'Person', 'Risk', 'Address']


## 2. Declare the Schema

Wrap each model in a `GraphNode` and specify identity fields.
Nested models become relation endpoints — `GraphRelation` names the edge.


In [ ]:
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema

project_node = GraphNode(node_class=Project, name_from="title", key_from="title", description="A project review")
company_node = GraphNode(node_class=Company, name_from="name", key_from="name", description="A client company")
person_node = GraphNode(node_class=Person, name_from="name", key_from="name", description="A team member")
risk_node = GraphNode(node_class=Risk, name_from="description", key_from="AUTO_ID", description="A project risk")

schema = GraphSchema(
    root_model_class=Project,
    nodes=[project_node, company_node, person_node, risk_node],
    relations=[
        GraphRelation(from_node=project_node, to_node=company_node, name="FOR_CLIENT"),
        GraphRelation(from_node=project_node, to_node=person_node, name="HAS_MEMBER"),
        GraphRelation(from_node=project_node, to_node=risk_node, name="HAS_RISK"),
    ],
)
print(f"Schema: {len(schema.nodes)} nodes, {len(schema.relations)} relations")

## 3. Inspect the Compiled Schema

`GraphSchema` auto-deduces field paths and excluded fields at construction time.

In [ ]:
print("=== Nodes ===")
for n in schema.nodes:
    fp = n.field_paths
    ex = n.excluded_fields
    print(f"  {n.label:15}  field_paths={fp!r}  excluded={ex!r}")

print("\n=== Relations ===")
for r in schema.relations:
    print(f"  {r.from_node.label} -[{r.name}]-> {r.to_node.label}")

print("\n=== Markdown table ===")
from genai_graph.kg.schema import ResolvedSchema

resolved = ResolvedSchema.from_graph_schema(schema)
print(resolved.to_markdown())

print("\n=== Validation warnings ===")
from genai_graph.kg.schema.compiler import validate_schema_coherence

warnings = validate_schema_coherence(schema)
print(warnings if warnings else "None — schema is clean")

## 4. Visualise the Schema

Interactive D3.js diagram of node types and relationship types.

In [ ]:
schema_html = resolved.to_html()
show_html_in_notebook(schema_html, "schema")
print(f"Schema HTML: {len(schema_html):,} bytes")

## 5. Ingest Data

`create_graph()` walks the root model, splits fields into nodes and relations,
and upserts everything into the graph database.


In [ ]:
from genai_graph.kg.ingest import create_graph, restart_database

backend = restart_database()  # fresh in-memory Ladybug DB

sample_projects = [
    Project(
        title="Cloud Migration",
        status="in-progress",
        client=Company(name="Acme Corp", sector="Retail", hq=Address(city="Paris")),
        team=[
            Person(name="Alice Martin", role="Lead"),
            Person(name="Bob Chen", role="Engineer"),
        ],
        risks=[
            Risk(description="Data loss during migration", impact="high"),
            Risk(description="Timeline overrun", impact="medium"),
        ],
    ),
    Project(
        title="ERP Modernisation",
        status="planning",
        client=Company(name="GlobalSoft", sector="Finance"),
        team=[Person(name="Alice Martin", role="Lead"), Person(name="Carol Li", role="Architect")],
        risks=[Risk(description="Budget overrun", impact="high")],
    ),
    Project(
        title="Data Platform",
        status="active",
        client=Company(name="Acme Corp", sector="Retail", hq=Address(city="Paris")),
        team=[Person(name="Bob Chen", role="Engineer")],
        risks=[],
    ),
]

for project in sample_projects:
    create_graph(backend, project, schema)

print(f"Ingested {len(sample_projects)} projects")
for label in ["Project", "Company", "Person", "Risk"]:
    df = backend.execute_get_as_df(f"MATCH (n:{label}) RETURN count(n) AS cnt")
    print(f"  {label}: {df['cnt'].iloc[0]}")

## 6. Query the KG

Standard Cypher — identical syntax to Neo4j.

In [ ]:
import pandas as pd


def run(cypher: str, title: str = "") -> None:
    df = backend.execute_get_as_df(cypher)
    if title:
        print(f"\n--- {title} ---")
    print(df.to_string(index=False) if not df.empty else "(no results)")


# Projects and their clients
run(
    "MATCH (p:Project)-[:FOR_CLIENT]->(c:Company) RETURN p.title, p.status, c.name AS client",
    "Projects by client",
)

# Team members per project
run(
    "MATCH (p:Project)-[:HAS_MEMBER]->(m:Person) RETURN p.title, m.name, m.role ORDER BY p.title",
    "Team members",
)

# High-impact risks
run(
    "MATCH (p:Project)-[:HAS_RISK]->(r:Risk) WHERE r.impact = 'high' RETURN p.title, r.description",
    "High-impact risks",
)

# Persons involved in multiple projects
run(
    """
    MATCH (p:Project)-[:HAS_MEMBER]->(m:Person)
    WITH m.name AS person, count(DISTINCT p) AS projects
    WHERE projects > 1
    RETURN person, projects ORDER BY projects DESC
    """,
    "Persons in multiple projects",
)

## 7. Visualise the KG Data

Interactive D3.js graph of the actual nodes and edges in the database.

In [ ]:
from genai_graph.kg.export.html import generate_html

kg_html = generate_html(connection=backend)
show_html_in_notebook(kg_html, "kg_data")
print(f"KG HTML: {len(kg_html):,} bytes, ~{kg_html.count('"id"')} nodes")